In [1]:
# Cell 1: imports & paths
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import Descriptors, QED
from bs4 import BeautifulSoup

ROOT = Path("..")          # this notebook is in notebooks/
DATA_DIR = ROOT / "data"

PUBCHEM_PATH = DATA_DIR / "pubchem.csv"
MOLEC_PATH = DATA_DIR / "molecule_classification_dataset.csv"
DRUGBANK_XML = DATA_DIR / "drugbank.xml"

PUBCHEM_CLEAN = DATA_DIR / "pubchem_clean.csv"
MOLEC_CLEAN = DATA_DIR / "molecule_classification_clean.csv"
DRUGBANK_PARSED = DATA_DIR / "drugbank_parsed.csv"

print("PUBCHEM:", PUBCHEM_PATH.resolve())
print("MOLEC:", MOLEC_PATH.resolve())
print("DRUGBANK XML:", DRUGBANK_XML.resolve())


PUBCHEM: C:\Users\notan\OneDrive\Desktop\genai-drug-discovery-rag\data\PubChem.csv
MOLEC: C:\Users\notan\OneDrive\Desktop\genai-drug-discovery-rag\data\molecule_classification_dataset.csv
DRUGBANK XML: C:\Users\notan\OneDrive\Desktop\genai-drug-discovery-rag\data\drugbank.xml


In [2]:
# Cell 2: load CSVs & detect SMILES columns

def safe_read_csv(path):
    print(f"Loading {path} ...")
    df = pd.read_csv(path, low_memory=False)
    print("Shape:", df.shape)
    return df

pubchem_df = safe_read_csv(PUBCHEM_PATH)
molec_df = safe_read_csv(MOLEC_PATH)

def detect_smiles_col(df):
    candidates = ["smiles", "SMILES", "canonical_smiles", "isomeric_smiles", "smiles_string"]
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    # heuristic fallback: look for SMILES-like strings
    for c in df.columns:
        sample = df[c].dropna().astype(str)
        if len(sample) == 0:
            continue
        s = sample.iloc[0]
        if any(ch in s for ch in ["(", ")", "=", "#", "/", "\\", "@", "[", "]"]):
            return c
    return None

pubchem_smiles_col = detect_smiles_col(pubchem_df)
molec_smiles_col = detect_smiles_col(molec_df)

print("Detected SMILES column in pubchem:", pubchem_smiles_col)
print("Detected SMILES column in molecule_classification_dataset:", molec_smiles_col)


Loading ..\data\pubchem.csv ...
Shape: (196627, 38)
Loading ..\data\molecule_classification_dataset.csv ...
Shape: (8711, 10)
Detected SMILES column in pubchem: SMILES
Detected SMILES column in molecule_classification_dataset: SMILES


In [3]:
# Cell 3: validate SMILES & compute MolWeight + QED

def validate_and_props(smiles_series):
    rows = []
    for s in tqdm(smiles_series.fillna("").astype(str), desc="Validating SMILES"):
        s = s.strip()
        valid = False
        mw = None
        qed = None
        if s:
            mol = Chem.MolFromSmiles(s)
            if mol is not None:
                valid = True
                mw = Descriptors.MolWt(mol)
                qed = QED.qed(mol)
        rows.append((s, valid, mw, qed))
    return pd.DataFrame(rows, columns=["SMILES", "RDKit_valid", "MolWeight", "QED"])


# ---- PubChem ----
if pubchem_smiles_col is None:
    raise ValueError("No SMILES column detected in pubchem.csv – please inspect the file and update data_ingestion.")

pubchem_min = pubchem_df[[pubchem_smiles_col]].rename(columns={pubchem_smiles_col: "SMILES"})
pubchem_min["SMILES"] = pubchem_min["SMILES"].astype(str).str.strip()

pubchem_valid = validate_and_props(pubchem_min["SMILES"])
pubchem_clean = pd.concat([pubchem_min.reset_index(drop=True), pubchem_valid[["RDKit_valid", "MolWeight", "QED"]]], axis=1)

print("PubChem: total rows:", len(pubchem_clean))
print("PubChem: valid SMILES:", pubchem_clean["RDKit_valid"].sum())

pubchem_clean = (
    pubchem_clean[pubchem_clean["RDKit_valid"]]
    .drop_duplicates(subset=["SMILES"])
    .reset_index(drop=True)
)

print("PubChem after filtering valid & dedupe:", len(pubchem_clean))


# ---- Molecule classification dataset ----
if molec_smiles_col is not None:
    molec_min = molec_df[[molec_smiles_col]].rename(columns={molec_smiles_col: "SMILES"})
    molec_min["SMILES"] = molec_min["SMILES"].astype(str).str.strip()

    molec_valid = validate_and_props(molec_min["SMILES"])
    molec_clean = pd.concat([molec_min.reset_index(drop=True), molec_valid[["RDKit_valid", "MolWeight", "QED"]]], axis=1)

    print("Molecule classification: total rows:", len(molec_clean))
    print("Molecule classification: valid SMILES:", molec_clean["RDKit_valid"].sum())

    molec_clean = (
        molec_clean[molec_clean["RDKit_valid"]]
        .drop_duplicates(subset=["SMILES"])
        .reset_index(drop=True)
    )
    print("Molecule classification after filtering:", len(molec_clean))
else:
    molec_clean = pd.DataFrame(columns=["SMILES", "RDKit_valid", "MolWeight", "QED"])
    print("No SMILES column detected in molecule_classification_dataset.csv")


Validating SMILES:   0%|          | 0/196627 [00:00<?, ?it/s]

[23:59:43] Explicit valence for atom # 1 Br, 3, is greater than permitted
[23:59:43] Explicit valence for atom # 1 Cl, 3, is greater than permitted
[23:59:43] WARNING: not removing hydrogen atom without neighbors
[23:59:43] WARNING: not removing hydrogen atom without neighbors
[23:59:43] Explicit valence for atom # 1 Si, 8, is greater than permitted
[23:59:43] Explicit valence for atom # 3 Si, 8, is greater than permitted
[23:59:43] Explicit valence for atom # 1 Si, 8, is greater than permitted
[23:59:43] WARNING: not removing hydrogen atom without neighbors
[23:59:43] WARNING: not removing hydrogen atom without neighbors
[23:59:45] Explicit valence for atom # 1 Br, 3, is greater than permitted
[23:59:45] WARNING: not removing hydrogen atom without neighbors
[23:59:45] WARNING: not removing hydrogen atom without neighbors
[23:59:46] WARNING: not removing hydrogen atom without neighbors
[23:59:46] WARNING: not removing hydrogen atom without neighbors
[23:59:46] WARNING: not removing hyd

PubChem: total rows: 196627
PubChem: valid SMILES: 196301
PubChem after filtering valid & dedupe: 196185


Validating SMILES:   0%|          | 0/8711 [00:00<?, ?it/s]

Molecule classification: total rows: 8711
Molecule classification: valid SMILES: 8711
Molecule classification after filtering: 8709


In [4]:
# Cell 4: save cleaned PubChem & molecule classification datasets

print("Saving cleaned PubChem to:", PUBCHEM_CLEAN)
pubchem_clean.to_csv(PUBCHEM_CLEAN, index=False)

print("Saving cleaned molecule classification dataset to:", MOLEC_CLEAN)
molec_clean.to_csv(MOLEC_CLEAN, index=False)

print("✅ Saved cleaned CSVs.")


Saving cleaned PubChem to: ..\data\pubchem_clean.csv
Saving cleaned molecule classification dataset to: ..\data\molecule_classification_clean.csv
✅ Saved cleaned CSVs.


In [5]:
# Stream-parse DrugBank XML safely and write parsed rows to CSV
# Paste into a notebook cell or run as a script.
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm.auto import tqdm
import csv
from rdkit import Chem
import sys

# === Config ===
NOTEBOOK_LOC = Path(".")  # set to "." if running from project root; if running from notebooks/, use Path("..")
DATA_DIR = NOTEBOOK_LOC / ".." / "data" if (NOTEBOOK_LOC / ".." / "data").exists() else Path("data")
DRUGBANK_PATH = DATA_DIR / "drugbank.xml"
OUT_CSV = DATA_DIR / "drugbank_parsed.csv"

BATCH_WRITE = 1000         # number of records to buffer before writing to CSV
VALIDATE_SMILES = False    # set True to validate SMILES using RDKit during parse (slower)

# Safety checks
if not DRUGBANK_PATH.exists():
    raise FileNotFoundError(f"DrugBank file not found at: {DRUGBANK_PATH.resolve()}")

print("Parsing DrugBank from:", DRUGBANK_PATH)
print("Output will be written to:", OUT_CSV)
print("Batch write size:", BATCH_WRITE, "Validate SMILES:", VALIDATE_SMILES)

# Helper functions (namespace-robust)
def find_text(elem, tag_localname):
    """Return text of first child whose local-name equals tag_localname (ignores namespace)."""
    # direct children
    for child in elem:
        # tag may be like '{namespace}localname' or 'localname'
        if child.tag.endswith("}" + tag_localname) or child.tag == tag_localname:
            return child.text
    # fallback: search deeper with findall and test localname
    for child in elem.iter():
        t = child.tag
        if t.endswith("}" + tag_localname) or t == tag_localname:
            return child.text
    return None

def findall_elems(elem, path_local):
    """Find all elements by a simple local-name based path (e.g., 'calculated-properties/property')."""
    parts = path_local.split("/")
    # naive recursive search
    results = []
    def rec(current, idx):
        if idx >= len(parts):
            results.append(current)
            return
        tag = parts[idx]
        for child in current:
            if child.tag.endswith("}" + tag) or child.tag == tag:
                rec(child, idx+1)
    rec(elem, 0)
    return results

# Prepare CSV: overwrite if exists
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["drugbank_ids","name","SMILES","inchi","description","targets","RDKit_valid"])
    writer.writeheader()

# Stream-parse
context = ET.iterparse(str(DRUGBANK_PATH), events=("end",))
buffer = []
count = 0
# iterparse yields many events; we'll pick elements whose tag localname == 'drug'
for event, elem in tqdm(context, desc="Streaming DrugBank XML"):
    tag = elem.tag
    if tag.endswith("}drug") or tag == "drug":
        # parse this <drug> element
        try:
            # drugbank-ids (may be several)
            db_ids = []
            for id_tag in elem.findall(".//"):
                # iterate children and pick drugbank-id localname matches
                if id_tag.tag.endswith("}drugbank-id") or id_tag.tag == "drugbank-id":
                    if id_tag.text:
                        db_ids.append(id_tag.text.strip())
            # name
            name = find_text(elem, "name") or ""
            name = name.strip() if name else ""

            # description (may be long)
            description = find_text(elem, "description") or ""
            description = description.strip()

            # calculated properties -> search for property/kind/value pairs
            smiles = None
            inchi = None
            # find property elements under calculated-properties
            # We'll inspect children of elem for any 'calculated-properties' parent first
            calc_props = []
            # Try local-name based approach
            for child in elem:
                if child.tag.endswith("}calculated-properties") or child.tag == "calculated-properties":
                    for prop in child:
                        calc_props.append(prop)
            # fallback: search deeper if not found
            if not calc_props:
                for node in elem.iter():
                    if node.tag.endswith("}calculated-properties") or node.tag == "calculated-properties":
                        for prop in node:
                            calc_props.append(prop)
            # parse each property element for kind/value
            for prop in calc_props:
                # find kind and value under prop
                kind_text = None
                value_text = None
                for sub in prop:
                    if sub.tag.endswith("}kind") or sub.tag == "kind":
                        kind_text = sub.text
                    if sub.tag.endswith("}value") or sub.tag == "value":
                        value_text = sub.text
                if kind_text and value_text:
                    kk = kind_text.strip().lower()
                    if "smiles" in kk and not smiles:
                        smiles = value_text.strip()
                    if "inchi" in kk and not inchi:
                        inchi = value_text.strip()

            # targets: collect polypeptide names or external identifiers
            targets = []
            # find targets parent(s)
            for tnode in elem.iter():
                if tnode.tag.endswith("}targets") or tnode.tag == "targets":
                    for t in tnode:
                        # look for polypeptide inside target
                        for sub in t.iter():
                            if sub.tag.endswith("}polypeptide") or sub.tag == "polypeptide":
                                # prefer polypeptide name
                                pname = find_text(sub, "name")
                                if pname:
                                    targets.append(pname.strip())
                                else:
                                    # find external-identifier/identifier
                                    for ext in sub.iter():
                                        if ext.tag.endswith("}external-identifier") or ext.tag == "external-identifier":
                                            ident = find_text(ext, "identifier")
                                            if ident:
                                                targets.append(ident.strip())
            # simple RDKit validation flag (optional)
            rdkit_ok = ""
            if VALIDATE_SMILES and smiles:
                try:
                    m = Chem.MolFromSmiles(smiles)
                    rdkit_ok = bool(m)
                except Exception:
                    rdkit_ok = False

            row = {
                "drugbank_ids": "|".join(db_ids) if db_ids else "",
                "name": name,
                "SMILES": smiles or "",
                "inchi": inchi or "",
                "description": description,
                "targets": "|".join(targets) if targets else "",
                "RDKit_valid": rdkit_ok
            }
            buffer.append(row)
            count += 1
        except Exception as e:
            # log and skip problematic entry
            print("Parse error for one entry:", e, file=sys.stderr)

        # write buffered rows periodically
        if len(buffer) >= BATCH_WRITE:
            with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=["drugbank_ids","name","SMILES","inchi","description","targets","RDKit_valid"])
                for r in buffer:
                    writer.writerow(r)
            buffer = []

        # clear the processed element to free memory
        elem.clear()

# write any leftover buffer
if buffer:
    with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["drugbank_ids","name","SMILES","inchi","description","targets","RDKit_valid"])
        for r in buffer:
            writer.writerow(r)
    buffer = []

print(f"Done parsing. Total drug entries processed: {count}")
print(f"Parsed CSV saved to: {OUT_CSV.resolve()}")


Parsing DrugBank from: ..\data\drugbank.xml
Output will be written to: ..\data\drugbank_parsed.csv
Batch write size: 1000 Validate SMILES: False


Streaming DrugBank XML: 0it [00:00, ?it/s]

Done parsing. Total drug entries processed: 65233
Parsed CSV saved to: C:\Users\notan\OneDrive\Desktop\genai-drug-discovery-rag\data\drugbank_parsed.csv


In [6]:
# Cell 6: quick summary

print("✅ Data ingestion complete.")
print("Saved files:")
print(" -", PUBCHEM_CLEAN)
print(" -", MOLEC_CLEAN)
print(" -", DRUGBANK_PARSED)


✅ Data ingestion complete.
Saved files:
 - ..\data\pubchem_clean.csv
 - ..\data\molecule_classification_clean.csv
 - ..\data\drugbank_parsed.csv
